# 🚀 03: Controlled Retrieval, Oracle Labels, and Learned Eviction

*Part of the KV-Cache Eviction Capstone Series*
*Estimated time: 30–60 minutes*

---

This notebook is the core development workflow for our learned eviction policy. It uses the canonical controlled retrieval corpus, generates oracle importance labels using attention and ablation, trains a 28-input MLP scorer, and selects the best checkpoint using strictly held-out validation data.

LongBench is reserved for a future external-validity extension and is not used for the current training, validation, or locked-test claims.

### Experiment Roadmap
```text
[Controlled corpus] → [Oracle labels] → [Train] → [Validation] → [Frozen checkpoint]
```


## Step 1: Why Does This Matter?

Machine learning is notoriously vulnerable to data leakage. If our learned policy sees the test data during training, or if we select our checkpoint based on test-set performance, our results will be an illusion.

**Learning and Leakage Controls:**
- We assign examples to deterministic 60/20/20 train/validation/test partitions.
- We fit feature normalization **on train only**.
- We use validation NDCG@3 for checkpoint selection.
- We **do not run** benchmark test rows here. They are locked away until Notebook 04.


In [ ]:
NOTEBOOK_ID = '03_rtx_pro6000_controlled_learned_eviction'
REQUESTED_PROFILE = 'rtx_pro_6000'

# Colab bootstrap: install pinned dependencies and unpack the shared core.
# Upload kvcore_bundle.zip supplied with this notebook suite if kvcore is not present.
from pathlib import Path
import sys, subprocess, zipfile

PINNED = [
    'transformers==4.56.2', 'accelerate==1.10.1', 'datasets==4.0.0',
    'huggingface_hub==0.34.4', 'bitsandbytes==0.47.0', 'safetensors==0.6.2',
    'sentencepiece==0.2.1', 'scipy==1.16.1', 'matplotlib==3.10.6',
    'seaborn==0.13.2', 'pandas==2.3.2',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PINNED])

if not Path('kvcore').exists():
    try:
        from google.colab import files
        print('Upload kvcore_bundle.zip from the delivered suite.')
        uploaded = files.upload()
        archive = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
        if archive is None:
            raise FileNotFoundError('Please upload kvcore_bundle.zip.')
        with zipfile.ZipFile(archive) as zf:
            zf.extractall('.')
    except ImportError as error:
        raise RuntimeError('Run in Google Colab or place the kvcore directory beside this notebook.') from error

sys.path.insert(0, str(Path('.').resolve()))
from kvcore import *
from kvcore.config import BENCHMARKS, MODELS, POLICY_DEFAULTS, PROFILES, SUITE_VERSION
print({'suite_version': SUITE_VERSION, 'ruler_revision': BENCHMARKS['ruler']['revision'], 'longbench_revision': BENCHMARKS['longbench']['revision']})


In [ ]:
import json, os, platform, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

REQUESTED_PROFILE = REQUESTED_PROFILE  # defined by the notebook title cell
NOTEBOOK_ID = globals().get('NOTEBOOK_ID', 'runtime')
set_all_seeds(590)
profile, runtime_status = select_profile(REQUESTED_PROFILE)
run_root = ensure_run_root(f'kv_eviction_{NOTEBOOK_ID}')
manifest = run_manifest(
    run_root,
    notebook=NOTEBOOK_ID,
    requested_profile=REQUESTED_PROFILE,
    active_profile=profile.name,
    model=model_spec(profile.model_tier),
    runtime_status=runtime_status,
)
print(json.dumps({'notebook': NOTEBOOK_ID, 'requested_profile': REQUESTED_PROFILE, 'active_profile': profile.name, 'runtime_status': runtime_status, 'run_root': str(run_root)}, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required for model execution. In Colab: Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


In [ ]:
# Model and tokenizer are always loaded at immutable Hub revisions from kvcore.config.
# If the runtime is smaller than the requested tier, runtime_status records the fallback.
model, tokenizer = load_model_and_tokenizer(model_spec(profile.model_tier), load_mode=profile.load_mode, attn_implementation='sdpa')
print({'model': model.config._name_or_path, 'requested_profile': REQUESTED_PROFILE, 'active_profile': profile.name, 'load_mode': profile.load_mode})


In [ ]:
RUN_LABEL_GENERATION = False
RUN_TRAINING = False
RUN_VALIDATION_EVAL = False

all_rows = load_controlled_retrieval(tokenizer=tokenizer, seed=590)
corpus_audit = audit_controlled_retrieval_corpus(all_rows)
assert corpus_audit['passed'], corpus_audit
partitions = pd.DataFrame(all_rows)[['benchmark','task','row_id','partition','planned_context_length','answer_location_band']]
partitions.to_csv(run_root / 'manifests' / 'controlled_retrieval_partitions.csv', index=False)
print(partitions.groupby(['planned_context_length','partition']).size().unstack(fill_value=0))
train_rows = [row for row in all_rows if row['partition'] == 'train']
validation_rows = [row for row in all_rows if row['partition'] == 'validation']
test_rows = [row for row in all_rows if row['partition'] == 'test']
assert len(train_rows) == 120 and len(validation_rows) == 40 and len(test_rows) == 40


## Step 2: Oracle Labels from Model Evidence

How do we know which tokens are actually important? We measure it. We generate labels by combining attention evidence with the actual increase in loss (Negative Log-Likelihood) caused by ablating (removing) that block from the cache. These are self-generated model labels.


In [ ]:
import gc

from kvcore.labels import label_blocks, save_labels
from kvcore.evaluation import answer_ids, tokenise_prompt

label_rows = []
if RUN_LABEL_GENERATION:
    # Create labels from both development partitions. The locked test partition
    # is deliberately excluded so validation-only checkpoint selection remains valid.
    LABEL_TRAIN_PER_LENGTH = 3
    LABEL_VALIDATION_PER_LENGTH = 2
    # Eager attention is used only to generate short-context oracle features.
    # A fixed cap bounds the peak quadratic attention allocation per document.
    LABEL_PROFILE_CONTEXT_CAP = 1024
    LABEL_MAX_COUNTERFACTUAL_BLOCKS = 4
    label_inputs = []
    label_memory_audit = []
    for source_rows, partition_name, per_length in (
        (train_rows, 'train', LABEL_TRAIN_PER_LENGTH),
        (validation_rows, 'validation', LABEL_VALIDATION_PER_LENGTH),
    ):
        for planned_length in CONTROLLED_RETRIEVAL['context_lengths']:
            candidates = [row for row in source_rows if row['planned_context_length'] == planned_length]
            chosen = candidates[:per_length]
            if len(chosen) != per_length:
                raise RuntimeError(f'Expected {per_length} {partition_name} records at length {planned_length}, found {len(chosen)}.')
            label_inputs.extend(chosen)
    label_input_summary = pd.DataFrame(label_inputs).groupby(['planned_context_length', 'partition']).size()
    print('Label-document selection by length and partition:\n', label_input_summary)
    assert sum(row['partition'] == 'train' for row in label_inputs) == 12
    assert sum(row['partition'] == 'validation' for row in label_inputs) == 8
    assert not any(row['partition'] == 'test' for row in label_inputs)

    for document_index, row in enumerate(label_inputs, start=1):
        prefix_ids = answer = profile_meta = blocks = None
        try:
            prefix_ids = tokenise_prompt(tokenizer, row['prompt'], LABEL_PROFILE_CONTEXT_CAP, next(model.parameters()).device)
            answer = answer_ids(tokenizer, row.get('answers', []), next(model.parameters()).device)
            profile_meta = profile_short_context(
                model, prefix_ids, min(prefix_ids.shape[-1], LABEL_PROFILE_CONTEXT_CAP), run_root,
                tag=f"label_{row['task']}_{row['row_id']}",
                prefill_chunk_tokens=min(profile.prefill_chunk_tokens, LABEL_PROFILE_CONTEXT_CAP),
            )
            if not profile_meta['attention_available']:
                raise RuntimeError('Eager attention was unavailable; do not substitute fabricated attention labels.')
            blocks = label_blocks(
                model, prefix_ids, answer, profile_meta['profile_path'], block_size=16,
                max_counterfactual_blocks=LABEL_MAX_COUNTERFACTUAL_BLOCKS,
                attention_weight=0.5, prefill_chunk_tokens=min(profile.prefill_chunk_tokens, LABEL_PROFILE_CONTEXT_CAP),
            )
            for item in blocks:
                item.update({
                    'example_id': f"{row['task']}::{row['row_id']}", 'task': row['task'],
                    'source_partition': row['partition'], 'planned_context_length': row['planned_context_length'],
                    'label_profile_context_tokens': int(prefix_ids.shape[-1]),
                })
            label_rows.extend(blocks)
        finally:
            # The label rows hold CPU features only. Release all per-document
            # model tensors before moving to the next eager-attention profile.
            prefix_ids = answer = profile_meta = blocks = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                free_bytes, total_bytes = torch.cuda.mem_get_info()
                audit_row = {
                    'document_index': document_index, 'row_id': row['row_id'],
                    'partition': row['partition'], 'planned_context_length': row['planned_context_length'],
                    'free_gib_after_cleanup': round(free_bytes / 2**30, 3),
                    'total_gib': round(total_bytes / 2**30, 3),
                    'allocated_gib': round(torch.cuda.memory_allocated() / 2**30, 3),
                    'reserved_gib': round(torch.cuda.memory_reserved() / 2**30, 3),
                }
                label_memory_audit.append(audit_row)
                print(f"Label document {document_index}/{len(label_inputs)} complete; free VRAM after cleanup: {audit_row['free_gib_after_cleanup']:.2f} GiB")
    label_path = save_labels(label_rows, run_root / 'labels' / 'oracle_block_labels.pt', {
        'label_definition': '0.5 attention + 0.5 NLL impact', 'block_size': 16,
        'source': 'controlled_train_validation_short_context_oracle',
        'label_profile_context_cap': LABEL_PROFILE_CONTEXT_CAP,
        'max_counterfactual_blocks': LABEL_MAX_COUNTERFACTUAL_BLOCKS,
        'label_documents': {'train': 12, 'validation': 8, 'test': 0},
    })
    write_json(run_root / 'manifests' / 'label_document_selection.json', {
        'train_per_length': LABEL_TRAIN_PER_LENGTH, 'validation_per_length': LABEL_VALIDATION_PER_LENGTH,
        'label_profile_context_cap': LABEL_PROFILE_CONTEXT_CAP,
        'max_counterfactual_blocks': LABEL_MAX_COUNTERFACTUAL_BLOCKS,
        'rows': [{'row_id': row['row_id'], 'partition': row['partition'], 'planned_context_length': row['planned_context_length']} for row in label_inputs],
    })
    write_json(run_root / 'runtime' / 'label_generation_memory_audit.json', label_memory_audit)
    print('Saved', len(label_rows), 'labels to', label_path)
else:
    print('Labels are intentionally not generated until RUN_LABEL_GENERATION=True; no pseudo-labels are fabricated.')


## Step 3: Learned 28-Input Block Ranker

We train a lightweight MLP to predict the oracle importance score from 28 easily computed attention and positional features. The training loop uses pairwise logistic ranking loss to learn which block to keep when forced to choose between two.


In [ ]:
from kvcore.scorer import ScorerConfig, train_scorer, load_scorer, split_documents, ranking_metrics

TRAIN_CONFIG_GRID = [
    {'learning_rate': 1e-3, 'dropout': 0.10, 'weight_decay': 1e-4, 'block_size': 16},
    {'learning_rate': 3e-4, 'dropout': 0.10, 'weight_decay': 1e-4, 'block_size': 16},
    {'learning_rate': 1e-3, 'dropout': 0.20, 'weight_decay': 1e-4, 'block_size': 16},
]
# Validation-only model selection. Do not inspect held-out test ranking metrics here.
if RUN_TRAINING:
    if not label_rows:
        import torch
        label_rows = torch.load(run_root / 'labels' / 'oracle_block_labels.pt', map_location='cpu')['labels']
    trials = []
    best = None
    for trial_index, params in enumerate(TRAIN_CONFIG_GRID):
        trial_root = run_root / 'trials' / f'trial_{trial_index:02d}'
        config = ScorerConfig(learning_rate=params['learning_rate'], dropout=params['dropout'], weight_decay=params['weight_decay'])
        scorer, normaliser, summary = train_scorer(label_rows, config=config, run_root=trial_root)
        record = {'trial': trial_index, **params, 'validation_ndcg_at_3': summary['best_validation_ndcg_at_3'], 'checkpoint': str(trial_root / 'checkpoints' / 'learned_kv_scorer.pt')}
        trials.append(record)
        if best is None or record['validation_ndcg_at_3'] > best['validation_ndcg_at_3']:
            best = record
    pd.DataFrame(trials).to_csv(run_root / 'results' / 'validation_model_selection.csv', index=False)
    write_json(run_root / 'checkpoints' / 'selected_by_validation_only.json', best)
    print('Selected checkpoint from validation only:', best)
else:
    print('Training disabled until RUN_TRAINING=True. This preserves a no-fabrication execution path.')


## Step 4: Validation-Only Policy Evaluation

Let us see how our newly trained policy compares to the baselines.

### What to Look For
Crucially, this evaluation runs **only on the validation partition**. If the learned policy outperforms the heuristics here, we have a strong candidate for the locked test.

**Next step:** Save the `run_root` directory, especially the `selected_by_validation_only.json` checkpoint. Notebook 04 requires it.


In [ ]:
if RUN_VALIDATION_EVAL:
    selected = json.loads((run_root / 'checkpoints' / 'selected_by_validation_only.json').read_text())
    learned, normaliser, payload = load_scorer(selected['checkpoint'])
    assert_checkpoint_reload(learned, normaliser, selected['checkpoint'], load_scorer)
    specs = [
        {'type': 'full'}, {'type': 'fifo'}, {'type': 'sink_recent'}, {'type': 'h2o'},
        {'type': 'learned_block', 'block_size': 16},
    ]
    records = evaluate_rows(model, tokenizer, validation_rows, specs, profile.budgets, profile.max_context_tokens, profile.decode_tokens, profile.prefill_chunk_tokens, run_root, learned_model=learned, normaliser=normaliser, artifact_stem='rtx_validation_only')
else:
    records = [requires_gpu_record('RTX PRO 6000 or equivalent', 'Set RUN_VALIDATION_EVAL=True', 'Validation policy evaluation is user-triggered.')]
pd.DataFrame(records).to_csv(run_root / 'results' / 'rtx_validation_only.csv', index=False)
